# Joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-05-joins.ipynb)

A join combines rows from two inputs by matching keys. Choosing the join type decides which unmatched rows
survive, and the **cardinality** of the keys (one-to-one, one-to-many, many-to-many) decides how many rows you
end up with. Most join bugs are cardinality bugs: a number that should be 1,150 comes out as 1,600.

Previous: [CASE expressions](sql-04-case-expressions.ipynb). Next: [window functions](sql-06-window-functions.ipynb).

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## The example data

`customers` and `orders` as before, plus two small extra tables: `customer_tags` (a customer can have several
tags) and `staff` (employees with a manager, for a self-join).

In [ ]:
create_view(
    "customers",
    "customer_id INT, name STRING, region STRING, signup_date DATE",
    [
        (1, "Acme Corp", "North", date(2022, 1, 5)),
        (2, "Beta LLC", "North", date(2022, 2, 10)),
        (3, "Cyan Inc", "South", date(2022, 3, 15)),
        (4, "Delta Co", "South", date(2022, 4, 20)),
        (5, "Echo Ltd", "West", date(2022, 5, 25)),
        (6, "Foxtrot GmbH", "West", date(2022, 6, 30)),
        (7, "Gamma SA", "East", date(2023, 1, 10)),
        (8, "Hotel BV", "East", date(2023, 2, 15)),
    ],
)

create_view(
    "orders",
    "order_id INT, customer_id INT, status STRING, amount INT, order_date DATE",
    [
        (1, 1, "completed", 100, date(2023, 1, 5)),
        (2, 1, "completed", 200, date(2023, 2, 10)),
        (3, 1, "completed", 150, date(2023, 3, 15)),
        (4, 2, "completed", 300, date(2023, 1, 20)),
        (5, 2, "cancelled", 250, date(2023, 2, 5)),
        (6, 2, "completed", 400, date(2023, 3, 20)),
        (7, 3, "completed", 120, date(2023, 1, 25)),
        (8, 3, "completed", 180, date(2023, 2, 28)),
        (9, 4, "pending", 220, date(2023, 2, 14)),
        (10, 4, "completed", 320, date(2023, 3, 30)),
        (11, 5, "completed", 500, date(2023, 1, 2)),
        (12, 5, "completed", 450, date(2023, 3, 5)),
        (13, 6, "completed", 275, date(2023, 2, 18)),
        (14, 6, "cancelled", 325, date(2023, 3, 25)),
    ],
)

In [ ]:
create_view(
    "customer_tags",
    "customer_id INT, tag STRING",
    [(1, "vip"), (1, "eu"), (2, "vip")],
)
create_view(
    "staff",
    "emp_id INT, name STRING, manager_id INT",
    [(1, "Alice", None), (2, "Bob", 1), (3, "Carol", 1), (4, "Dave", 2), (5, "Eve", 2)],
)

## Join types at a glance

| Join | Keeps |
| --- | --- |
| `INNER JOIN` | rows with a match on both sides |
| `LEFT JOIN` | every left row; right columns are `NULL` when there is no match |
| `RIGHT JOIN` | every right row (the mirror of `LEFT`) |
| `FULL OUTER JOIN` | every row from both sides |
| `LEFT SEMI JOIN` | left rows that have at least one match; right columns are not returned |
| `LEFT ANTI JOIN` | left rows that have no match |

`SEMI` and `ANTI` are the Spark spellings of `EXISTS` and `NOT EXISTS`. They never multiply rows.

### Inner join

Customers joined to their orders. Each order matches exactly one customer, so the 14 orders produce 14 rows.

In [ ]:
check(
    """
    SELECT COUNT(*) AS joined_rows
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    """,
    [(14,)],
    "inner join keeps every matched order",
)
check(
    """
    SELECT c.name, o.order_id
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    ORDER BY o.order_id
    LIMIT 3
    """,
    [("Acme Corp", 1), ("Acme Corp", 2), ("Acme Corp", 3)],
    "first three joined rows",
)

### Left join keeps the unmatched rows

Customers 7 and 8 have never ordered. A `LEFT JOIN` keeps them with `NULL` order columns. An inner join would
drop them silently.

In [ ]:
check(
    """
    SELECT c.customer_id, o.order_id
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE c.customer_id >= 7
    ORDER BY c.customer_id
    """,
    [(7, None), (8, None)],
    "left join keeps customers without orders",
)

### Filter in `ON` versus `WHERE`

This is the subtlety that catches people. A condition on the right-hand table in `WHERE` removes the
`NULL`-padded rows, which turns the outer join back into an inner join. The same condition in `ON` only
decides which right rows can match, so unmatched left rows survive.

In [ ]:
check(
    """
    SELECT c.customer_id, COUNT(o.order_id) AS completed_orders
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
    GROUP BY c.customer_id
    ORDER BY c.customer_id
    """,
    [(1, 3), (2, 2), (3, 2), (4, 1), (5, 2), (6, 1), (7, 0), (8, 0)],
    "condition in ON keeps every customer",
)
check(
    """
    SELECT c.customer_id, COUNT(o.order_id) AS completed_orders
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id
    ORDER BY c.customer_id
    """,
    [(1, 3), (2, 2), (3, 2), (4, 1), (5, 2), (6, 1)],
    "condition in WHERE drops customers 7 and 8",
)

## Cardinality: the join that multiplies a sum

`customer_tags` has two rows for customer 1. Joining orders to it repeats each of customer 1's orders once per
tag, so a `SUM` over the joined rows double-counts them. The fix is to avoid the one-to-many join when you only
need to test membership: a semi join (or `EXISTS`) keeps each order once.

Customer 1's completed orders total 450 and customer 2's total 700, so the correct answer for both customers is
1,150.

In [ ]:
check(
    """
    SELECT SUM(o.amount) AS naive_total
    FROM orders o
    JOIN customer_tags t ON t.customer_id = o.customer_id
    WHERE o.status = 'completed'
    """,
    [(1600,)],
    "naive join double-counts customer 1",
)
check(
    """
    SELECT SUM(o.amount) AS correct_total
    FROM orders o
    LEFT SEMI JOIN customer_tags t ON t.customer_id = o.customer_id
    WHERE o.status = 'completed'
    """,
    [(1150,)],
    "semi join counts each order once",
)

## Self-join

A table can join to itself when each side gets its own alias. Here each employee joins to their manager.
`LEFT JOIN` keeps the top of the hierarchy, Alice, whose manager is `NULL`.

In [ ]:
check(
    """
    SELECT e.name AS employee, m.name AS manager
    FROM staff e
    LEFT JOIN staff m ON m.emp_id = e.manager_id
    ORDER BY e.emp_id
    """,
    [("Alice", None), ("Bob", "Alice"), ("Carol", "Alice"), ("Dave", "Bob"), ("Eve", "Bob")],
    "self-join: employee and manager",
)

## Anti and semi joins

"Customers with no orders" is an anti join. "Customers with at least one order" is a semi join. Both return
customer columns only, and neither can duplicate a customer.

In [ ]:
check(
    """
    SELECT c.customer_id
    FROM customers c
    LEFT ANTI JOIN orders o ON o.customer_id = c.customer_id
    ORDER BY c.customer_id
    """,
    [(7,), (8,)],
    "anti join: customers with no orders",
)
check(
    """
    SELECT c.customer_id
    FROM customers c
    LEFT SEMI JOIN orders o ON o.customer_id = c.customer_id
    ORDER BY c.customer_id
    """,
    [(1,), (2,), (3,), (4,), (5,), (6,)],
    "semi join: customers with at least one order",
)

## Takeaways

- Choose the join type from what must survive: `LEFT` keeps the left side, `ANTI` keeps the unmatched side.
- Before joining, state the cardinality of each key. A one-to-many join repeats the "one" side's measures.
- Put conditions on the right-hand table in `ON` when you want unmatched left rows to survive.
- Prefer semi or anti joins to joins when you only need membership.